# Stellarator current drive

The current a neutral beam drives in Wendelstein 7-X, with the electrons' response on 3D flux surfaces. The fast ions carry a current $j_f$ along $\mathbf{B}$; the electrons shield it, $j = j_f\,[1 - Z_b / Z_\mathrm{eff}\,(1 - G)]$, and the trapped electrons, which cannot follow, reduce the shielding by $G$. In a tokamak $G$ is Sauter's $L_{31}$ of the trapped fraction $f_t$ and the collisionality (Sauter, Angioni and Lin-Liu, Phys. Plasmas 6, 2834, 1999). On 3D surfaces `slow_down` returns a NaN current by default; `trapped_electron_correction="sauter"` takes $L_{31}$ with the **effective trapped fraction** of the 3D surface,

$$
f_t = 1 - \frac{3}{4}\langle B^2\rangle \int_0^{1/B_\mathrm{max}} \frac{\lambda\,d\lambda}{\langle\sqrt{1 - \lambda B}\rangle},
$$

with the flux-surface averages $\langle\cdot\rangle$ over the 3D surface (`FluxSurfaces.trapped_fraction`; Lin-Liu and Miller, Phys. Plasmas 2, 1666, 1995, and DESC's `"trapped fraction"`).

**This is an approximation.** $L_{31}(f_t)$ is the axisymmetric coefficient; in a stellarator the trapped electrons' response, like the bootstrap current, also depends on how $|B|$ varies on the surface, not only on $f_t$ (Shaing and Callen, Phys. Fluids 26, 3315, 1983), and that geometric factor is not included. The collisionality takes $q = 1/|\iota|$, the toroidally averaged $R$ and, in place of the inverse aspect ratio, $\epsilon = (B_\mathrm{max} - B_\mathrm{min}) / (B_\mathrm{max} + B_\mathrm{min})$ (`FluxSurfaces.field_epsilon`), as Landreman, Buller and Drevlak (Phys. Plasmas, 2022) do for the bootstrap current of stellarators. The fast ions slow down on their birth surface with their local pitch (no orbit average in 3D, no helically trapped ions, no losses). The beam geometry is the stand-in of the *W7-X deposition* example, and the plasma is made up: hydrogen with 1 % carbon ($Z_\mathrm{eff} = 1.3$), $T_{e0} = 2$ keV.

## Equilibrium and shells

DESC's W7-X example, sampled once on a coarse grid for the deposition (`GriddedEquilibrium.cached` keeps the samples outside the repository), and ten shells in $\rho_\mathrm{tor}$ from DESC's flux coordinates on a $24 \times 24$ grid per field period.

In [ ]:
import os
import time
import warnings
from pathlib import Path

import numpy as np
import plasma_plots as pp
import xarray as xr
from desc.grid import LinearGrid

import freeofcharge as foc
from freeofcharge.desc_equilibrium import DESCEquilibrium
from freeofcharge.injectors import w7x_injectors
from freeofcharge.slowing_down import sauter_trapped_electron_correction, slow_down

pp.set_backend("plotly")  # interactive figures, embedded in the docs
pp.set_math("latex")  # formulas in the figures typeset as LaTeX
warnings.filterwarnings("ignore", category=UserWarning, module="desc")

start = time.perf_counter()
cache = Path(
    os.environ.get("FREEOFCHARGE_CACHE", Path.home() / ".cache" / "freeofcharge")
)
desc_w7x = DESCEquilibrium.from_example("W7-X")
w7x = foc.GriddedEquilibrium.cached(
    cache / "w7x_desc_64_32_64.nc", desc_w7x, (64, 32, 64)
)
rho_edges = np.linspace(0.0, 1.0, 11)
surfaces = w7x.flux_surfaces(rho_edges, field_points=(24, 24), radial_points=4)
print(f"equilibrium and shells in {time.perf_counter() - start:.0f} s")

## Effective trapped fraction

$f_t$ of each shell's centre surface, against DESC's own `"trapped fraction"` on the same surfaces and the tokamak estimate $1.46\sqrt{\epsilon}$ from the inverse aspect ratio $\epsilon$ (Start and Cordey's fit depends on $\epsilon$ alone). DESC's value agrees to a few $10^{-3}$, the difference of its coarser $B_\mathrm{max}$. In W7-X $f_t$ is far above the tokamak estimate, about 0.3 already near the axis, where $\epsilon \to 0$: the mirror (toroidal) variation of $|B|$ along the field lines traps electrons on every surface, so an $\epsilon$-based correction would miss most of them. The mirror $\epsilon$ of $|B|$ (`field_epsilon`) stays finite there too, which is why the collisionality uses it.

In [ ]:
eq = desc_w7x.equilibrium
grid = LinearGrid(rho=surfaces.rho, M=eq.M_grid, N=eq.N_grid, NFP=eq.NFP)
desc_ft = grid.compress(
    np.asarray(eq.compute("trapped fraction", grid=grid)["trapped fraction"])
)
shells = surfaces.to_dataset()
trapped = xr.concat(
    [
        shells.trapped_fraction,
        xr.DataArray(desc_ft, coords={"rho": shells.rho}),
        1.46 * np.sqrt(shells.inverse_aspect_ratio),
    ],
    dim=xr.DataArray(
        ["freeofcharge", "DESC", r"tokamak $1.46\sqrt{\epsilon}$"], dims="estimate"
    ),
)
trapped.attrs.update(long_name="trapped fraction $f_t$")
print(
    f"largest difference to DESC: {np.max(np.abs(surfaces.trapped_fraction / desc_ft - 1)):.1e}"
)
shells["field_epsilon"] = ("rho", surfaces.field_epsilon)
print(
    shells[["trapped_fraction", "inverse_aspect_ratio", "field_epsilon"]]
    .to_pandas()
    .round(3)
)
trapped.plasma.plot.profiles(x="rho", over="estimate", cmap="tab10")

## Driven current with and without the correction

The two OP1.2b sources (NI21, S1 and S2) at $n_{e0} = 4 \times 10^{19}\,\mathrm{m^{-3}}$, 500 markers per source and energy component. The births of all components slow down on their shells (`orbit_average=False`, the local pitch), and the driven current is computed three ways: the default in 3D (NaN, with a warning), `"none"` ($G = 0$: the shielding of a uniform field, $j = j_f (1 - 1/Z_\mathrm{eff})$), and `"sauter"` ($G = L_{31}$ with the effective $f_t$). The sign says the direction: negative is against $\mathbf{B}$.

In [ ]:
op12b = w7x_injectors(sources={"NI21": [1, 2]})
ions = {"H": (1, 1, 0.94), "C": (12, 6, 0.01)}  # Z_eff = 1.3


def drive(ne0, markers=500):
    plasma = foc.Plasma.parabolic(ne0, 2e3, ions=ions, radial_coordinate="toroidal")
    births = foc.deposit_sources(
        op12b, w7x, plasma, markers=markers, seed=1, step=0.02, rho_edges=rho_edges
    ).total()
    steady = {
        name: slow_down(
            births,
            plasma,
            surfaces,
            orbit_average=False,
            trapped_electron_correction=name,
        )
        for name in ("none", "sauter")
    }
    return plasma, births, steady


plasma, births, steady = drive(4e19)
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    default = slow_down(births, plasma, surfaces, orbit_average=False)
print(f"default in 3D: driven current {default.driven_current} ({caught[0].message})")
for name, result in steady.items():
    print(f"{name:>6}: driven current {result.driven_current / 1e3:+.1f} kA")
print(f"absorbed {births.absorbed_power / 1e6:.2f} MW")

The correction $G$ of each shell: Sauter's $L_{31}$ grows with $f_t$ and falls with the electron collisionality $\nu_e^*$, which is large at the cold, dense edge, so $G$ peaks at mid-radius. Near the axis $G$ stays near 0.2: with the geometric $\epsilon \to 0$ in place of `field_epsilon` the collisionality would diverge there and $G$ vanish.

In [ ]:
g = sauter_trapped_electron_correction(surfaces, plasma)
correction = xr.DataArray(
    g,
    coords={"rho": shells.rho},
    name="G",
    attrs={"long_name": "trapped-electron correction $G = L_{31}$"},
)
print(correction.to_pandas().round(3))
correction.plasma.plot.lineout(x="rho")

The current density profiles: the fast-ion current, and the net current with and without the correction. The correction raises the net current by the factor $[1 - (1 - G)/Z_\mathrm{eff}] / (1 - 1/Z_\mathrm{eff})$ in each shell, here 1.5 to 2.0: at $Z_\mathrm{eff} = 1.3$ the uncorrected shielding cancels 77 % of the fast-ion current, so the share of trapped electrons matters a lot.

In [ ]:
profiles = xr.concat(
    [
        steady["none"].to_dataset(surfaces).fast_ion_current_density,
        steady["none"].to_dataset(surfaces).current_density,
        steady["sauter"].to_dataset(surfaces).current_density,
    ],
    dim=xr.DataArray(
        ["fast ions", "net, $G = 0$", "net, Sauter with $f_t$"], dims="current"
    ),
)
profiles.attrs.update(long_name="parallel current density", units="A/m^2")
ratio = steady["sauter"].current_density / steady["none"].current_density
print(f"ratio corrected / uncorrected: {ratio.min():.2f} to {ratio.max():.2f}")
profiles.plasma.plot.profiles(x="rho", over="current", cmap="tab10")

## Total current against density

The central density from $10^{19}$ to $10^{20}\,\mathrm{m^{-3}}$, the same temperature. A denser plasma absorbs more of the beam, but its fast ions slow down faster (the slowing-down time falls as $T_e^{3/2}/n_e$) and its electrons are more collisional, which lowers $G$: the driven current falls with density, with and without the correction, and the correction matters less at high density. Fast-ion losses, not included here, would lower the current most at low density, where the slowing down takes longest.

In [ ]:
densities = [1e19, 2e19, 4e19, 7e19, 1e20]
totals = {"none": [], "sauter": []}
absorbed = []
for ne0 in densities:
    _, deposited, result = drive(ne0)
    absorbed.append(deposited.absorbed_power)
    for name in totals:
        totals[name].append(result[name].driven_current)
scan = xr.Dataset(
    {
        "driven_current": (
            ("correction", "ne0"),
            np.array([totals["none"], totals["sauter"]]) / 1e3,
            {"long_name": "driven current", "units": "kA"},
        ),
        "absorbed_power": (
            "ne0",
            np.array(absorbed) / 1e6,
            {"long_name": "absorbed power", "units": "MW"},
        ),
    },
    coords={
        "ne0": ("ne0", densities, {"long_name": r"$n_{e0}$", "units": "m^-3"}),
        "correction": ["$G = 0$", "Sauter with $f_t$"],
    },
)
print(scan.driven_current.to_pandas().T.round(1))
print(f"absorbed power [MW]: {np.round(scan.absorbed_power.values, 2)}")
scan.driven_current.plasma.plot.profiles(x="ne0", over="correction", cmap="tab10")

## What this does not include

- the stellarator geometric factor of the trapped-electron response (Shaing and Callen, 1983): $L_{31}(f_t)$ is the tokamak coefficient with the 3D $f_t$, which is the same approach as for the shielding in BEAMS3D for W7-X (Lazerson et al., Nucl. Fusion 61, 2021);
- the orbits of the fast ions: they slow down on their birth surface with their local pitch, without helically trapped ions or losses (the *Fast-ion orbits* example follows the 3D guiding-centre orbits);
- the real W7-X beam geometry and profiles: the injector boxes and the plasma are stand-ins, so the numbers are not a prediction for a W7-X discharge.